# NILM Approach 2 — Adaptive Per-Appliance Windows + Specialist Meta Model

This experiment keeps the five appliance-specific specialists and the 645-D meta fusion idea, but gives each specialist its own context-window length while keeping **the same center timestamps** for all five specialists.

Improvements included:
- Per-appliance window sizes chosen from train-only event-duration statistics.
- Same center timestamps across appliances, so stacking remains aligned.
- 12 static appliance-profile features + 6 dynamic window features.
- Per-appliance positive class weights.
- Focal loss for microwave and laptop (hard/rare examples).
- Early stopping + best-checkpoint saving.
- Meta training split into meta-train and threshold-validation so thresholds are not tuned on the same rows used to fit the meta model.
- F2 threshold for microwave, F1 thresholds for the other appliances.
- Test set is untouched until final evaluation.

**Important:** the window sizes below are starting candidates based on appliance behavior, not claimed optimal values. The notebook prints event-duration statistics so they can be audited and changed without changing the architecture.


In [ ]:
# Cell 1 — Imports and Google Drive
import os, json, math, random
import numpy as np, pandas as pd, tensorflow as tf

from google.colab import drive
drive.mount('/content/gdrive')

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Bidirectional, LSTM, GlobalAveragePooling1D,
    Dense, Dropout, Concatenate
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score

print("TensorFlow:", tf.__version__)


Mounted at /content/gdrive
TensorFlow: 2.20.0


In [ ]:
# Cell 2 — Configuration
DATA_PATH = "/content/gdrive/MyDrive/house_1/house_1_fully/house1_required_2years.csv"

SAVE_ROOT = "/content/gdrive/MyDrive/NILM_Approach2_2Years/adaptive_windows_focal_v3"
BASE_DIR = os.path.join(SAVE_ROOT, "specialist_models")
META_DIR = os.path.join(SAVE_ROOT, "meta_model")
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

APPLIANCES = ["laptop", "washing_machine", "tv", "fridge", "microwave"]

# Candidate design, expressed in samples. At ~6 s/sample:
# laptop  ~7.5 min, washing ~15 min, TV ~10 min, fridge ~10 min, microwave ~5 min.
WINDOW_SIZES = {
    "laptop": 75,
    "washing_machine": 150,
    "tv": 100,
    "fridge": 100,
    "microwave": 50,
}
# Keep the same center timestamps. A stride of 20 means predictions are ~2 minutes apart.
STRIDE = 20
BATCH_SIZE = 128

SPECIALIST_EPOCHS = 30
META_EPOCHS = 50
EARLY_STOP_PATIENCE = 6

# Per-appliance caps. These are deliberately not identical.
POS_WEIGHT_CAPS = {
    "laptop": 8.0,
    "washing_machine": 10.0,
    "tv": 10.0,
    "fridge": 3.0,
    "microwave": 15.0,
}

# Focal loss only where imbalance / hard examples are most problematic.
FOCAL_APPLIANCES = {"laptop", "microwave"}
FOCAL_GAMMA = 2.0

# Decision metric used for threshold selection on a held-out part of META.
BETA_BY_APPLIANCE = {
    "laptop": 1.0,
    "washing_machine": 1.0,
    "tv": 1.0,
    "fridge": 1.0,
    "microwave": 2.0,
}

print("Window sizes:", WINDOW_SIZES)
print("Stride:", STRIDE)
print("Save directory:", SAVE_ROOT)


Window sizes: {'laptop': 75, 'washing_machine': 150, 'tv': 100, 'fridge': 100, 'microwave': 50}
Stride: 20
Save directory: /content/gdrive/MyDrive/NILM_Approach2_2Years/adaptive_windows_focal_v3


In [ ]:
# Cell 3 — Load processed dataset
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


Shape: (9469092, 8)
Columns: ['datetime', 'timestamp', 'aggregate', 'laptop', 'washing_machine', 'tv', 'fridge', 'microwave']


In [ ]:
# Cell 4 — Clean data
required_cols = ["datetime", "timestamp", "aggregate"] + APPLIANCES
df = df[required_cols].copy()

for c in ["aggregate"] + APPLIANCES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["aggregate"] + APPLIANCES).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["datetime"])

aggregate = df["aggregate"].values.astype(np.float32)

TRAIN_CUTOFF = int(len(df) * 0.70)
SAMPLE_INTERVAL_S = float(df["datetime"].diff().dt.total_seconds().median())

print("Clean shape:", df.shape)
print(f"Train cutoff: {TRAIN_CUTOFF:,}")
print(f"Sample interval: {SAMPLE_INTERVAL_S:.2f} s")


Clean shape: (9469092, 8)
Train cutoff: 6,628,364
Sample interval: 6.00 s


In [ ]:
# Cell 5 — Per-appliance ON thresholds from TRAIN only
def auto_pick_threshold(power_values, min_threshold=3.0):
    vals = power_values[power_values > 0]
    if len(vals) < 50:
        return min_threshold
    log_vals = np.log1p(vals).reshape(-1, 1)
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(log_vals)
    centers = np.expm1(km.cluster_centers_.ravel())
    threshold = float(np.sqrt(max(centers.min(), 1e-6) * max(centers.max(), 1e-6)))
    return max(min_threshold, threshold)

ON_THRESHOLD = {}
for app in APPLIANCES:
    thr = auto_pick_threshold(df[app].values[:TRAIN_CUTOFF])
    ON_THRESHOLD[app] = thr
    print(f"{app:18s} threshold={thr:.2f} W")


laptop             threshold=4.65 W
washing_machine    threshold=57.04 W
tv                 threshold=10.17 W
fridge             threshold=148.40 W
microwave          threshold=28.70 W


In [ ]:
# Cell 6 — Create appliance ON/OFF labels
Y_state = np.zeros((len(df), len(APPLIANCES)), dtype=np.float32)
for j, app in enumerate(APPLIANCES):
    Y_state[:, j] = (df[app].values >= ON_THRESHOLD[app]).astype(np.float32)

for j, app in enumerate(APPLIANCES):
    on = int(Y_state[:, j].sum())
    off = len(Y_state) - on
    print(f"{app:18s} OFF={off:,} ON={on:,} ON%={100*on/len(Y_state):.2f}")


laptop             OFF=7,748,107 ON=1,720,985 ON%=18.17
washing_machine    OFF=9,127,683 ON=341,409 ON%=3.61
tv                 OFF=8,611,287 ON=857,805 ON%=9.06
fridge             OFF=9,352,786 ON=116,306 ON%=1.23
microwave          OFF=9,428,963 ON=40,129 ON%=0.42


In [ ]:
# Cell 7 — Profile knowledge features from TRAIN only
PROFILE_FEATURE_NAMES = [
    "typical_power_on", "delta_power_on_off", "standby_power", "power_std_on",
    "typical_on_duration_sec", "typical_off_duration_sec",
    "dominant_cycle_period_sec", "duty_cycle",
    "operations_per_day", "peak_hour_of_day", "hour_concentration",
    "weekday_weekend_ratio",
]

def event_runs(binary):
    binary = np.asarray(binary).astype(int)
    if binary.sum() == 0 or binary.sum() == len(binary):
        return np.array([]), np.array([])
    changes = np.diff(binary)
    starts = np.where(changes == 1)[0] + 1
    ends = np.where(changes == -1)[0] + 1
    if binary[0] == 1: starts = np.insert(starts, 0, 0)
    if binary[-1] == 1: ends = np.append(ends, len(binary))
    n = min(len(starts), len(ends))
    starts, ends = starts[:n], ends[:n]
    on_dur = ends - starts
    off_dur = starts[1:] - ends[:-1] if n > 1 else np.array([])
    return on_dur, off_dur

def compute_appliance_profile(df_train, app, threshold):
    power = df_train[app].values
    dt = df_train["datetime"]
    state = power >= threshold
    on_power, off_power = power[state], power[~state]
    typical_on = float(on_power.mean()) if len(on_power) else threshold * 2
    standby = float(off_power.mean()) if len(off_power) else 0.0
    delta = typical_on - standby
    std_on = float(on_power.std()) if len(on_power) > 1 else 0.0
    on_d, off_d = event_runs(state)
    typical_on_d = float(np.median(on_d) * SAMPLE_INTERVAL_S) if len(on_d) else SAMPLE_INTERVAL_S
    typical_off_d = float(np.median(off_d) * SAMPLE_INTERVAL_S) if len(off_d) else SAMPLE_INTERVAL_S
    cycle = typical_on_d + typical_off_d
    duty = float(state.mean())
    days = max((dt.iloc[-1] - dt.iloc[0]).total_seconds()/86400, 1e-6)
    ops_day = len(on_d) / days
    hours_on = dt[state].dt.hour
    if len(hours_on):
        mode_hour = int(hours_on.mode().iloc[0])
        hour_conc = float((hours_on == mode_hour).mean())
    else:
        mode_hour, hour_conc = 12, 0.0
    weekend = dt.dt.weekday >= 5
    weekday_frac = state[~weekend.values].mean() if (~weekend.values).sum() else 0.0
    weekend_frac = state[weekend.values].mean() if weekend.values.sum() else 0.0
    return np.array([
        typical_on, delta, standby, std_on, typical_on_d, typical_off_d,
        cycle, duty, ops_day, float(mode_hour), hour_conc,
        float(weekday_frac/(weekend_frac+1e-6))
    ], dtype=np.float32)

df_train_only = df.iloc[:TRAIN_CUTOFF]
PROFILE_MATRIX_RAW = np.stack([
    compute_appliance_profile(df_train_only, app, ON_THRESHOLD[app]) for app in APPLIANCES
])
profile_static_scaler = StandardScaler().fit(PROFILE_MATRIX_RAW)
PROFILE_MATRIX = profile_static_scaler.transform(PROFILE_MATRIX_RAW).astype(np.float32)

display(pd.DataFrame(PROFILE_MATRIX_RAW, index=APPLIANCES, columns=PROFILE_FEATURE_NAMES).round(2))


,typical_power_on,delta_power_on_off,standby_power,power_std_on,typical_on_duration_sec,typical_off_duration_sec,dominant_cycle_period_sec,duty_cycle,operations_per_day,peak_hour_of_day,hour_concentration,weekday_weekend_ratio
laptop,19.820000,19.809999,0.010000,9.040000,204.0,6.0,210.0,0.20,15.130000,9.0,0.07,2.05
washing_machine,677.070007,676.830017,0.230000,754.469971,12.0,6.0,18.0,0.03,87.709999,14.0,0.11,0.86
tv,104.500000,103.589996,0.900000,19.280001,54.0,6.0,60.0,0.09,62.930000,20.0,0.25,0.87
fridge,264.279999,231.979996,32.299999,140.289993,12.0,810.0,822.0,0.01,17.870001,1.0,0.05,0.99
microwave,1230.520020,1229.630005,0.890000,613.229980,30.0,960.0,990.0,0.00,5.650000,17.0,0.18,1.18


In [ ]:
# Cell 8 — Dynamic profile features
DYNAMIC_FEATURE_NAMES = [
    "window_mean_power", "window_max_power", "window_std_power",
    "window_duty_cycle", "deviation_from_typical_on", "window_largest_jump"
]
PROFILE_DIM = 18

def compute_dynamic_window_features(X_raw, threshold, typical_on):
    mean_p = X_raw.mean(axis=1)
    max_p = X_raw.max(axis=1)
    std_p = X_raw.std(axis=1)
    duty = (X_raw >= threshold).mean(axis=1)
    deviation = np.abs(mean_p - typical_on)
    jump = np.max(np.abs(np.diff(X_raw, axis=1)), axis=1) if X_raw.shape[1] > 1 else np.zeros(len(X_raw))
    return np.stack([mean_p, max_p, std_p, duty, deviation, jump], axis=1).astype(np.float32)

dyn_scalers = {}
def build_profile_input(X_raw, app, fit_scaler=False):
    j = APPLIANCES.index(app)
    typical_on = PROFILE_MATRIX_RAW[j, PROFILE_FEATURE_NAMES.index("typical_power_on")]
    dyn_raw = compute_dynamic_window_features(X_raw, ON_THRESHOLD[app], typical_on)
    if fit_scaler:
        dyn_scalers[app] = StandardScaler().fit(dyn_raw)
    dyn_scaled = dyn_scalers[app].transform(dyn_raw)
    static = np.repeat(PROFILE_MATRIX[j][None, :], len(X_raw), axis=0)
    return np.concatenate([static, dyn_scaled], axis=1).astype(np.float32)


In [ ]:
# Cell 9 — Event-duration report and per-appliance window sanity check
print(f"{'appliance':18s} {'median ON min':>14s} {'p90 ON min':>12s} {'chosen window min':>18s}")
for app in APPLIANCES:
    state = df_train_only[app].values >= ON_THRESHOLD[app]
    on_d, _ = event_runs(state)
    med = np.median(on_d)*SAMPLE_INTERVAL_S/60 if len(on_d) else np.nan
    p90 = np.percentile(on_d, 90)*SAMPLE_INTERVAL_S/60 if len(on_d) else np.nan
    chosen = WINDOW_SIZES[app]*SAMPLE_INTERVAL_S/60
    print(f"{app:18s} {med:14.2f} {p90:12.2f} {chosen:18.2f}")
print("\nThese chosen windows are deliberately shorter for short-event appliances and longer for long-cycle appliances.")


appliance           median ON min   p90 ON min  chosen window min
laptop                       3.40        40.10               7.50
washing_machine              0.20         1.00              15.00
tv                           0.90         2.60              10.00
fridge                       0.20         2.00              10.00
microwave                    0.50         1.20               5.00

These chosen windows are deliberately shorter for short-event appliances and longer for long-cycle appliances.


In [ ]:
# Cell 10 — Common center timestamps + appliance-specific raw windows
# All specialists predict at exactly the same center timestamps.
MAX_HALF = max(WINDOW_SIZES.values()) // 2
MIN_HALF = min(WINDOW_SIZES.values()) // 2
valid_centers = np.arange(MAX_HALF, len(df) - MAX_HALF, STRIDE, dtype=np.int64)

# chronological split by CENTER timestamp
N = len(valid_centers)
i1, i2 = int(0.70*N), int(0.85*N)
C_train, C_meta, C_test = valid_centers[:i1], valid_centers[i1:i2], valid_centers[i2:]

def extract_windows_at_centers(signal, centers, window_size):
    half = window_size // 2
    starts = centers - half
    ends = starts + window_size
    return np.stack([signal[s:e] for s,e in zip(starts, ends)]).astype(np.float32)

X_raw = {}
Y = {}
for app in APPLIANCES:
    ws = WINDOW_SIZES[app]
    X_raw[app] = {
        "train": extract_windows_at_centers(aggregate, C_train, ws),
        "meta": extract_windows_at_centers(aggregate, C_meta, ws),
        "test": extract_windows_at_centers(aggregate, C_test, ws),
    }
    j = APPLIANCES.index(app)
    Y[app] = {
        "train": Y_state[C_train, j],
        "meta": Y_state[C_meta, j],
        "test": Y_state[C_test, j],
    }

print("Common center counts:", len(C_train), len(C_meta), len(C_test))
for app in APPLIANCES:
    print(f"{app:18s} train window={X_raw[app]['train'].shape}, labels={Y[app]['train'].shape}")


Common center counts: 331413 71017 71018
laptop             train window=(331413, 75), labels=(331413,)
washing_machine    train window=(331413, 150), labels=(331413,)
tv                 train window=(331413, 100), labels=(331413,)
fridge             train window=(331413, 100), labels=(331413,)
microwave          train window=(331413, 50), labels=(331413,)


In [ ]:
# Cell 11 — Per-appliance scaling
X_scaled = {}
for app in APPLIANCES:
    scaler = StandardScaler()
    tr = X_raw[app]["train"]
    scaler.fit(tr.reshape(-1,1))
    X_scaled[app] = {
        "train": scaler.transform(tr.reshape(-1,1)).reshape(tr.shape).astype(np.float32),
        "meta": scaler.transform(X_raw[app]["meta"].reshape(-1,1)).reshape(X_raw[app]["meta"].shape).astype(np.float32),
        "test": scaler.transform(X_raw[app]["test"].reshape(-1,1)).reshape(X_raw[app]["test"].shape).astype(np.float32),
    }
    X_raw[app]["scaler"] = scaler
print("Scaling complete.")


Scaling complete.


In [ ]:
# Cell 12 — Focal loss + specialist architecture
def binary_focal_loss(gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0-tf.keras.backend.epsilon())
        p_t = y_true*y_pred + (1-y_true)*(1-y_pred)
        return tf.reduce_mean(-tf.pow(1.0-p_t, gamma) * (
            y_true*tf.math.log(y_pred) + (1-y_true)*tf.math.log(1-y_pred)
        ))
    return loss

def build_specialist_model(window_size, use_focal=False):
    agg_in = Input(shape=(window_size,1), name="aggregate_input")
    profile_in = Input(shape=(PROFILE_DIM,), name="appliance_profile_input")
    x = Conv1D(32,7,activation="relu")(agg_in)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(64,5,activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = GlobalAveragePooling1D()(x)
    k = Dense(64,activation="relu")(profile_in)
    k = Dropout(0.20)(k)
    fused = Concatenate()([x,k])
    fused = Dense(128,activation="relu")(fused)
    fused = Dropout(0.30)(fused)
    learned = Dense(128,activation="relu",name="learned_feature_128")(fused)
    out = Dense(1,activation="sigmoid",name="state_output")(learned)
    model = Model([agg_in,profile_in],out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=binary_focal_loss(FOCAL_GAMMA) if use_focal else "binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
    )
    return model


In [ ]:
# Cell 13 — Train/resume five specialists with appliance-specific windows and loss
specialist_models, feature_models, specialist_diagnostics = {}, {}, {}

print(f"{'appliance':18s} {'window':>8s} {'pos_frac':>10s} {'weight':>9s} {'loss':>10s}")
for j, app in enumerate(APPLIANCES):
    ws = WINDOW_SIZES[app]
    model_path = os.path.join(BASE_DIR, f"{app}_specialist_w{ws}.keras")
    profile_train = build_profile_input(X_raw[app]["train"], app, fit_scaler=True)
    profile_meta = build_profile_input(X_raw[app]["meta"], app)
    ytr, ymeta = Y[app]["train"], Y[app]["meta"]
    pos_frac = float(ytr.mean())
    raw_weight = (1-pos_frac)/max(pos_frac,1e-6)
    cap = POS_WEIGHT_CAPS[app]
    weight = min(raw_weight, cap)
    use_focal = app in FOCAL_APPLIANCES
    print(f"{app:18s} {ws:8d} {pos_frac:10.4f} {weight:9.2f} {('focal' if use_focal else 'BCE'):>10s}")

    need = True
    model = None
    if os.path.exists(model_path):
        try:
            model = load_model(model_path, custom_objects={"loss": binary_focal_loss(FOCAL_GAMMA),
                                                          "<lambda>": binary_focal_loss(FOCAL_GAMMA)})
            if model.input_shape[0][1] == ws and model.input_shape[1][-1] == PROFILE_DIM:
                need = False
                print("  Loaded:", model_path)
            else:
                print("  Existing checkpoint shape mismatch; retraining.")
        except Exception as e:
            print("  Could not load checkpoint; retraining:", str(e)[:160])
    if need:
        model = build_specialist_model(ws, use_focal)
        model.fit(
            [X_scaled[app]["train"][...,None], profile_train], ytr,
            validation_data=([X_scaled[app]["meta"][...,None], profile_meta], ymeta),
            epochs=SPECIALIST_EPOCHS, batch_size=BATCH_SIZE, verbose=1,
            class_weight=None if use_focal else {0:1.0,1:float(weight)},
            callbacks=[
                ModelCheckpoint(model_path, monitor="val_loss", save_best_only=True, verbose=1),
                EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=1)
            ]
        )
        # save the restored best weights explicitly
        model.save(model_path)
        print("  Saved:", model_path)
    specialist_models[app] = model
    feature_models[app] = Model(model.inputs, model.get_layer("learned_feature_128").output)
    specialist_diagnostics[app] = {
        "window_size": ws, "pos_frac": pos_frac, "raw_weight": raw_weight,
        "cap": cap, "weight": weight, "loss": "focal" if use_focal else "binary_crossentropy"
    }
print("All specialists ready.")


appliance            window   pos_frac    weight       loss
laptop                   75     0.1994      4.01      focal
  Loaded: /content/gdrive/MyDrive/NILM_Approach2_2Years/adaptive_windows_focal_v3/specialist_models/laptop_specialist_w75.keras
washing_machine         150     0.0345     10.00        BCE
  Loaded: /content/gdrive/MyDrive/NILM_Approach2_2Years/adaptive_windows_focal_v3/specialist_models/washing_machine_specialist_w150.keras
tv                      100     0.0895     10.00        BCE
  Loaded: /content/gdrive/MyDrive/NILM_Approach2_2Years/adaptive_windows_focal_v3/specialist_models/tv_specialist_w100.keras
fridge                  100     0.0119      3.00        BCE
Epoch 1/30
2589/2590 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9869 - loss: 0.1001 - precision: 0.4556 - recall: 0.2770
Epoch 1: val_loss improved from None to 0.03235, saving model to /content/gdrive/MyDrive/NILM_Approach2_2Years/adaptive_windows_focal_v3/specialist_models/fridge_specialist_w100.keras

In [ ]:
# Cell 14 — Specialist META diagnostics
specialist_meta_probs = {}
for app in APPLIANCES:
    prof = build_profile_input(X_raw[app]["meta"], app)
    p = specialist_models[app].predict([X_scaled[app]["meta"][...,None], prof], batch_size=BATCH_SIZE, verbose=0).ravel()
    specialist_meta_probs[app] = p
    y = Y[app]["meta"].astype(int)
    auc = roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan
    print(f"{app:18s} AUC={auc:.4f} F1@0.5={f1_score(y,(p>=.5).astype(int),zero_division=0):.4f}")


laptop             AUC=0.9258 F1@0.5=0.4391
washing_machine    AUC=0.9953 F1@0.5=0.8089
tv                 AUC=0.9767 F1@0.5=0.7328
fridge             AUC=0.9749 F1@0.5=0.7266
microwave          AUC=0.9955 F1@0.5=0.3852


In [ ]:
# Cell 15 — Build aligned specialist fusion inputs
def make_stack(split):
    feature_list, prob_list = [], []
    for app in APPLIANCES:
        prof = build_profile_input(X_raw[app][split], app)
        f = feature_models[app].predict(
            [X_scaled[app][split][...,None], prof], batch_size=BATCH_SIZE, verbose=0
        )
        p = specialist_models[app].predict(
            [X_scaled[app][split][...,None], prof], batch_size=BATCH_SIZE, verbose=0
        ).reshape(-1,1)
        feature_list.append(f)
        prob_list.append(p)
    return np.concatenate(feature_list+prob_list, axis=1).astype(np.float32)

META_INPUT_FULL = make_stack("meta")
TEST_INPUT = make_stack("test")
Y_meta_matrix = np.column_stack([Y[a]["meta"] for a in APPLIANCES]).astype(np.float32)
Y_test_matrix = np.column_stack([Y[a]["test"] for a in APPLIANCES]).astype(np.float32)
print("META_INPUT:", META_INPUT_FULL.shape)
print("TEST_INPUT:", TEST_INPUT.shape)


META_INPUT: (71017, 645)
TEST_INPUT: (71018, 645)


In [ ]:
# Cell 16 — Meta train / threshold-validation split
# Keep the last 20% of META as threshold-validation; it is not used to fit meta weights.
m = len(META_INPUT_FULL)
mt_end = int(0.80*m)
META_TRAIN = META_INPUT_FULL[:mt_end]
META_VAL = META_INPUT_FULL[mt_end:]
Y_META_TRAIN = Y_meta_matrix[:mt_end]
Y_META_VAL = Y_meta_matrix[mt_end:]
print(META_TRAIN.shape, META_VAL.shape)


(56813, 645) (14204, 645)


In [ ]:
# Cell 17 — Meta model
def build_meta_model(input_dim=645):
    inp = Input(shape=(input_dim,), name="specialist_fusion_input")
    x = Dense(256,activation="relu")(inp)
    x = Dropout(.30)(x)
    x = Dense(128,activation="relu")(x)
    x = Dropout(.20)(x)
    out = Dense(5,activation="sigmoid",name="final_state_output")(x)
    model = Model(inp,out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return model

META_MODEL_PATH = os.path.join(META_DIR,"approach2_adaptive_meta_model.keras")
if os.path.exists(META_MODEL_PATH):
    meta_model = load_model(META_MODEL_PATH)
    print("Loaded saved meta model.")
else:
    meta_model = build_meta_model(645)
    meta_model.fit(
        META_TRAIN,Y_META_TRAIN,
        validation_data=(META_VAL,Y_META_VAL),
        epochs=META_EPOCHS,batch_size=BATCH_SIZE,verbose=1,
        callbacks=[
            ModelCheckpoint(META_MODEL_PATH,monitor="val_loss",save_best_only=True,verbose=1),
            EarlyStopping(monitor="val_loss",patience=EARLY_STOP_PATIENCE,restore_best_weights=True,verbose=1)
        ]
    )
    meta_model.save(META_MODEL_PATH)
meta_model = load_model(META_MODEL_PATH)


In [ ]:
# Cell 18 — Per-appliance F1/F2 threshold tuning on META-VALIDATION only
def best_threshold(y_true,y_prob,beta=1.0):
    best_t,best_score=.5,-1
    for t in np.linspace(.02,.98,97):
        pred=(y_prob>=t).astype(int)
        score=fbeta_score(y_true,pred,beta=beta,zero_division=0)
        if score>best_score:
            best_t,best_score=float(t),float(score)
    return best_t,best_score

META_VAL_PROBS = meta_model.predict(META_VAL,batch_size=BATCH_SIZE,verbose=0)
BEST_THRESHOLDS={}
print(f"{'appliance':18s} {'beta':>6s} {'threshold':>10s} {'val score':>12s}")
for j,app in enumerate(APPLIANCES):
    beta=BETA_BY_APPLIANCE[app]
    t,score=best_threshold(Y_META_VAL[:,j].astype(int),META_VAL_PROBS[:,j],beta)
    BEST_THRESHOLDS[app]=t
    print(f"{app:18s} {beta:6.1f} {t:10.2f} {score:12.4f}")


In [ ]:
# Cell 19 — Final test prediction + evaluation
TEST_PROBS = meta_model.predict(TEST_INPUT,batch_size=BATCH_SIZE,verbose=1)
TEST_PRED=np.zeros_like(TEST_PROBS,dtype=int)
for j,app in enumerate(APPLIANCES):
    TEST_PRED[:,j]=(TEST_PROBS[:,j]>=BEST_THRESHOLDS[app]).astype(int)

rows=[]
for j,app in enumerate(APPLIANCES):
    y=Y_test_matrix[:,j].astype(int)
    p=TEST_PRED[:,j]
    auc=roc_auc_score(y,TEST_PROBS[:,j]) if len(np.unique(y))==2 else np.nan
    rows.append({
        "Appliance":app,
        "Window_samples":WINDOW_SIZES[app],
        "Window_minutes":WINDOW_SIZES[app]*SAMPLE_INTERVAL_S/60,
        "Accuracy":accuracy_score(y,p),
        "Precision":precision_score(y,p,zero_division=0),
        "Recall":recall_score(y,p,zero_division=0),
        "F1":f1_score(y,p,zero_division=0),
        "F2":fbeta_score(y,p,beta=2,zero_division=0),
        "Threshold":BEST_THRESHOLDS[app],
        "ROC_AUC":auc
    })
results=pd.DataFrame(rows)
display(results)


555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


,Appliance,Window_samples,Window_minutes,Accuracy,Precision,Recall,F1,F2,Threshold,ROC_AUC
0,laptop,75,7.5,0.765792,0.331160,0.525062,0.406155,0.470020,0.26,0.785699
1,washing_machine,150,15.0,0.984145,0.750964,0.924695,0.828823,0.883802,0.60,0.994912
2,tv,100,10.0,0.930539,0.584202,0.912608,0.712378,0.820374,0.28,0.970547
3,fridge,100,10.0,0.994607,0.835878,0.721186,0.774308,0.741535,0.31,0.975091
4,microwave,50,5.0,0.989749,0.489111,0.765625,0.596899,0.687851,0.32,0.992253


In [ ]:
# Cell 20 — Save results and complete experiment configuration
RESULT_PATH=os.path.join(SAVE_ROOT,"approach2_adaptive_windows_final_results.csv")
SUMMARY_PATH=os.path.join(SAVE_ROOT,"approach2_adaptive_windows_run_summary.json")
results.to_csv(RESULT_PATH,index=False)
summary={
    "window_sizes":WINDOW_SIZES,
    "stride":STRIDE,
    "sample_interval_s":SAMPLE_INTERVAL_S,
    "on_threshold":ON_THRESHOLD,
    "pos_weight_caps":POS_WEIGHT_CAPS,
    "focal_appliances":sorted(FOCAL_APPLIANCES),
    "focal_gamma":FOCAL_GAMMA,
    "beta_by_appliance":BETA_BY_APPLIANCE,
    "best_thresholds":BEST_THRESHOLDS,
    "specialist_diagnostics":specialist_diagnostics,
    "results":results.to_dict(orient="records")
}
with open(SUMMARY_PATH,"w") as f: json.dump(summary,f,indent=2)
print("Saved:",RESULT_PATH)
print("Saved:",SUMMARY_PATH)


## How to interpret this experiment

Compare these results against the previous tuned-threshold baseline:

- Microwave F1 ≈ 0.426
- Laptop F1 ≈ 0.401
- Fridge F1 ≈ 0.709
- TV F1 ≈ 0.716
- Washing machine F1 ≈ 0.819

The goal is **not** to assume every appliance improves. A successful experiment should show which appliance-specific changes helped and which did not.

The test set is used only once at the end. Thresholds are selected from the held-out part of META, not from TEST.
